# CipherMark — évaluation sur Google Colab

Exécute la chaîne de tests CipherMark : tests unitaires, chaîne crypto bout-à-bout,
et le protocole de robustesse du hash perceptuel avec **DINOv2** (le run décisif).

Exécuter les cellules dans l'ordre. GPU non obligatoire, mais accélère DINOv2
(`Exécution > Modifier le type d'exécution > T4 GPU`).

In [ ]:
# 1) Cloner le depot (branche crystal)
# Si le depot est prive : decommenter la ligne TOKEN et coller un token GitHub
# (Settings > Developer settings > Personal access tokens, portee 'repo').
REPO = "github.com/ngueagho/distseal-meta.git"
# import os; os.environ['GH_TOKEN'] = 'ghp_votre_token_ici'
import os
url = f"https://{os.environ['GH_TOKEN']}@{REPO}" if 'GH_TOKEN' in os.environ else f"https://{REPO}"
!git clone --branch crystal --depth 1 {url} distseal-meta
%cd distseal-meta

In [ ]:
# 2) Dependances (tout le reste est preinstalle sur Colab)
!pip install -q reedsolo

In [ ]:
# 3) Tests unitaires (attendu : 14/14 OK)
!PYTHONPATH=. python -m tests.ciphermark.run_all

In [ ]:
# 4) Chaine crypto bout-a-bout
#    (attendu : round-trip 20/20, replay 0/20, faux positifs 0/50)
!python -m scripts.ciphermark.gen_keys --out-dir ./keys
!PYTHONPATH=. python -m scripts.ciphermark.eval_ciphermark --n 20 --keys-dir ./keys

In [ ]:
# 5) Corpus de 28 images naturelles (photos scikit-image, 4 recadrages chacune)
import numpy as np
from PIL import Image
from skimage import data
import os

os.makedirs("corpus-test", exist_ok=True)
saved = 0
for name in ["astronaut", "chelsea", "coffee", "rocket", "camera", "hubble_deep_field", "cat"]:
    try:
        img = getattr(data, name)()
    except Exception as e:
        print(f"skip {name}: {e}")
        continue
    if img.ndim == 2:
        img = np.stack([img] * 3, axis=-1)
    H, W = img.shape[:2]
    s = min(H, W)
    for i, c in enumerate([img[:s, :s], img[:s, W-s:], img[H-s:, :s], img[H-s:, W-s:]]):
        Image.fromarray(c).resize((256, 256), Image.BILINEAR).save(f"corpus-test/{name}_{i}.png")
        saved += 1
print(saved, "images dans corpus-test/")

In [ ]:
# 6) LE RUN DECISIF : robustesse du canal hash avec DINOv2
#    (le telechargement des poids ~84 Mo prend quelques secondes sur Colab)
#    Lecture : colonne 'recuperable' de la phase 1 = fraction d'images dont le
#    hash reste corrigeable (<= 8 octets). C'est elle qui decide de la viabilite.
!mkdir -p results
!PYTHONPATH=. python -m scripts.ciphermark.eval_phash_robustness \
    --n 28 --data-dir ./corpus-test --csv results/phash_dino_reelles.csv

In [ ]:
# 6bis) Comparaison : meme protocole avec le fallback DCT (borne basse)
!PYTHONPATH=. python -m scripts.ciphermark.eval_phash_robustness \
    --n 28 --data-dir ./corpus-test --backbone dct --csv results/phash_dct_reelles.csv

In [ ]:
# 7) Si la recuperabilite DINOv2 est entre 50 et 90 % : re-essayer avec plus
#    de parite Reed-Solomon (32 octets -> corrige 16 octets d'erreur)
!PYTHONPATH=. python -m scripts.ciphermark.eval_phash_robustness \
    --n 28 --data-dir ./corpus-test --rs-nsym 32 --csv results/phash_dino_rs32.csv

In [ ]:
# 8) Recuperer les CSV (pour le chapitre 3 du memoire)
from google.colab import files
for f in ["results/phash_dino_reelles.csv", "results/phash_dct_reelles.csv", "results/phash_dino_rs32.csv"]:
    try:
        files.download(f)
    except Exception as e:
        print(f"{f}: {e}")